In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

FONDO = "#ffffff"
TEXTO = "#111827"
TEXTO_TENUE = "#6b7280"
REJILLA = "#e5e7eb"

COLOR = {
    "postgresql": "#00618a",
    "mysql": "#d97706",
    "python_postgresql": "#8a919e",
    "python_mysql": "#8a919e",
}

NOMBRE = {
    "postgresql": "PostgreSQL",
    "mysql": "MySQL",
    "python_postgresql": "Python",
    "python_mysql": "Python",
}

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.weight": "bold",
    "text.color": TEXTO,
    "axes.labelcolor": TEXTO,
    "xtick.color": TEXTO,
    "ytick.color": TEXTO,
    "figure.facecolor": FONDO,
    "axes.facecolor": FONDO,
    "savefig.facecolor": FONDO,
    "axes.titleweight": "bold",
    "axes.titlecolor": TEXTO,
    "figure.dpi": 110,
})

IMGS = "imgs"
os.makedirs(IMGS, exist_ok=True)


def guardar(figura, nombre):
    figura.savefig(os.path.join(IMGS, nombre), dpi=150, bbox_inches="tight")


def sin_marco(ax, dejar=()):
    for lado, espina in ax.spines.items():
        espina.set_visible(lado in dejar)
        if lado in dejar:
            espina.set_color(TEXTO_TENUE)

In [ ]:
CARGAS = ["1k", "10k", "100k"]

FILAS_POR_TABLA = {"1k": 1_000, "10k": 10_000, "100k": 100_000}

MOTORES = ["postgresql", "mysql"]

PYTHON_DE = {"postgresql": "python_postgresql", "mysql": "python_mysql"}

TIEMPOS = pd.DataFrame(
    [
        ("insercion",    "postgresql",        239.662, 2527.523, 29055.644),
        ("consulta_1",   "postgresql",          2.584,    7.187,     51.956),
        ("consulta_2",   "postgresql",          8.585,   85.430,   1128.943),
        ("traer_tablas", "python_postgresql", 102.735,  157.031,   1290.734),
        ("consulta_1",   "python_postgresql",   5.231,    8.992,     97.092),
        ("consulta_2",   "python_postgresql",   5.361,   18.298,    223.609),
        ("insercion",    "mysql",             457.290, 3955.518, 40036.582),
        ("consulta_1",   "mysql",               1.140,    8.942,     88.969),
        ("consulta_2",   "mysql",               7.494,   77.024,    902.467),
        ("traer_tablas", "python_mysql",      247.296,  388.768,   3703.509),
        ("consulta_1",   "python_mysql",       25.575,   10.136,    100.473),
        ("consulta_2",   "python_mysql",       13.681,   22.631,    239.285),
    ],
    columns=["paso", "motor", *CARGAS],
)


def tiempos_de(paso, motor):
    fila = TIEMPOS[(TIEMPOS["paso"] == paso) & (TIEMPOS["motor"] == motor)]
    if fila.empty:
        return np.full(len(CARGAS), np.nan)
    return fila[CARGAS].to_numpy(dtype=float).ravel()


def formato(ms):
    if np.isnan(ms):
        return "—"
    if ms < 10:
        return f"{ms:.1f} ms"
    if ms < 1000:
        return f"{ms:.0f} ms"
    if ms < 60_000:
        return f"{ms / 1000:.1f} s"
    return f"{ms / 60_000:.1f} min"


def etiqueta_de_carga(carga):
    return f"{FILAS_POR_TABLA[carga]:,} filas".replace(",", ".")


TIEMPOS

In [ ]:
figura, ax = plt.subplots(figsize=(10, 5.5))

posiciones = np.arange(len(CARGAS))
ancho = 0.72 / len(MOTORES)
tope = max(np.nanmax(tiempos_de("insercion", motor)) for motor in MOTORES)

for indice, motor in enumerate(MOTORES):
    valores = tiempos_de("insercion", motor)
    desfase = (indice - (len(MOTORES) - 1) / 2) * ancho
    ax.bar(posiciones + desfase, valores, ancho * 0.84, color=COLOR[motor],
           label=NOMBRE[motor], zorder=2)
    for x, valor in zip(posiciones + desfase, valores):
        ax.text(x, valor + tope * 0.02, formato(valor), ha="center", va="bottom",
                fontsize=13, color=TEXTO)

ax.set_xticks(posiciones)
ax.set_xticklabels([etiqueta_de_carga(c) for c in CARGAS], fontsize=13)
ax.tick_params(axis="x", length=0, pad=12)
ax.set_yticks([])
ax.set_xlim(-0.6, len(CARGAS) - 0.4)
ax.set_ylim(0, tope * 1.18)
sin_marco(ax, dejar=("top",))

ax.legend(loc="upper left", frameon=False, fontsize=12)
ax.set_title("Tiempo de inserción por tamaño de carga", fontsize=18, pad=22)
figura.tight_layout()
guardar(figura, "insercion.png")
plt.show()

In [ ]:
figura, ax = plt.subplots(figsize=(10, 6))

posiciones = np.arange(len(CARGAS))
TRAZO = {"consulta_1": (0, (5, 2)), "consulta_2": "solid"}

for motor in MOTORES:
    for paso in ("consulta_1", "consulta_2"):
        valores = tiempos_de(paso, motor)
        ax.plot(posiciones, valores, color=COLOR[motor], linewidth=2,
                linestyle=TRAZO[paso], marker="o", markersize=8,
                markerfacecolor=COLOR[motor], markeredgecolor=FONDO, markeredgewidth=2,
                label=f"{NOMBRE[motor]} · {paso.replace('_', ' ')}", zorder=3)
        ax.text(posiciones[-1] + 0.06, valores[-1], f"  {formato(valores[-1])}",
                color=TEXTO, fontsize=12, va="center")

ax.set_yscale("log")
ax.set_xticks(posiciones)
ax.set_xticklabels([etiqueta_de_carga(c) for c in CARGAS], fontsize=13)
ax.set_xlim(-0.25, len(CARGAS) - 0.3)
ax.tick_params(axis="y", labelsize=11, colors=TEXTO_TENUE)
ax.grid(axis="y", color=REJILLA, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
sin_marco(ax, dejar=("left", "bottom"))

ax.legend(loc="upper left", frameon=False, fontsize=11, labelcolor=TEXTO, ncol=2)
ax.set_title("Tiempo de las consultas en SQL", fontsize=18, pad=22)
figura.tight_layout()
guardar(figura, "consultas_sql.png")
plt.show()

In [ ]:
def comparar_motores(paso):
    figura, ax = plt.subplots(figsize=(9.5, 5.5))

    posiciones = np.arange(len(CARGAS))
    ancho = 0.72 / len(MOTORES)
    por_motor = {motor: tiempos_de(paso, motor) for motor in MOTORES}
    tope = max(np.nanmax(valores) for valores in por_motor.values())

    for indice, motor in enumerate(MOTORES):
        desfase = (indice - (len(MOTORES) - 1) / 2) * ancho
        ax.bar(posiciones + desfase, por_motor[motor], ancho * 0.84,
               color=COLOR[motor], label=NOMBRE[motor], zorder=2)
        for x, valor in zip(posiciones + desfase, por_motor[motor]):
            ax.text(x, valor + tope * 0.02, formato(valor), ha="center", va="bottom",
                    fontsize=12, color=TEXTO)

    ax.set_xticks(posiciones)
    ax.set_xticklabels([etiqueta_de_carga(c) for c in CARGAS], fontsize=13)
    ax.tick_params(axis="x", length=0, pad=12)
    ax.set_yticks([])
    ax.set_xlim(-0.6, len(CARGAS) - 0.4)
    ax.set_ylim(0, tope * 1.18)
    sin_marco(ax, dejar=("top",))

    ax.legend(loc="upper left", frameon=False, fontsize=12)
    ax.set_title(f"{paso.replace('_', ' ').capitalize()} — PostgreSQL contra MySQL",
                 fontsize=18, pad=22)
    figura.tight_layout()
    guardar(figura, f"motores_{paso}.png")
    plt.show()


for paso in ("consulta_1", "consulta_2"):
    comparar_motores(paso)

In [ ]:
def comparar_con_python(paso, motor):
    en_python = PYTHON_DE[motor]
    figura, ax = plt.subplots(figsize=(9.5, 5.5))

    posiciones = np.arange(len(CARGAS))
    series = [(motor, tiempos_de(paso, motor)),
              (en_python, tiempos_de(paso, en_python))]
    ancho = 0.72 / len(series)
    tope = max(np.nanmax(valores) for _, valores in series)

    for indice, (clave, valores) in enumerate(series):
        desfase = (indice - (len(series) - 1) / 2) * ancho
        ax.bar(posiciones + desfase, valores, ancho * 0.84, color=COLOR[clave],
               label=NOMBRE[clave], zorder=2)
        for x, valor in zip(posiciones + desfase, valores):
            ax.text(x, valor + tope * 0.02, formato(valor), ha="center", va="bottom",
                    fontsize=12, color=TEXTO)

    ax.set_xticks(posiciones)
    ax.set_xticklabels([etiqueta_de_carga(c) for c in CARGAS], fontsize=13)
    ax.tick_params(axis="x", length=0, pad=12)
    ax.set_yticks([])
    ax.set_xlim(-0.6, len(CARGAS) - 0.4)
    ax.set_ylim(0, tope * 1.18)
    sin_marco(ax, dejar=("top",))

    ax.legend(loc="upper left", frameon=False, fontsize=12)
    ax.set_title(f"{paso.replace('_', ' ').capitalize()} — {NOMBRE[motor]} contra Python",
                 fontsize=18, pad=22)
    figura.tight_layout()
    guardar(figura, f"{paso}_{motor}_vs_python.png")
    plt.show()


for motor in MOTORES:
    for paso in ("consulta_1", "consulta_2"):
        comparar_con_python(paso, motor)

In [ ]:
SERIES_GENERALES = [
    ("postgresql", "PostgreSQL", None),
    ("mysql", "MySQL", None),
    ("python_postgresql", "Python · desde PostgreSQL", None),
    ("python_mysql", "Python · desde MySQL", "///"),
]

totales = {clave: tiempos_de("consulta_1", clave) + tiempos_de("consulta_2", clave)
           for clave, _, _ in SERIES_GENERALES}

figura, paneles = plt.subplots(1, len(CARGAS), figsize=(13, 4.4))

posiciones = np.arange(len(SERIES_GENERALES))[::-1]

for panel, (ax, carga) in enumerate(zip(paneles, CARGAS)):
    valores = [totales[clave][panel] for clave, _, _ in SERIES_GENERALES]
    tope = max(valores)

    for posicion, (clave, _, trama), valor in zip(posiciones, SERIES_GENERALES, valores):
        ax.barh(posicion, valor, 0.62, color=COLOR[clave], hatch=trama,
                edgecolor=FONDO, linewidth=1.2, zorder=2)
        ax.text(valor + tope * 0.04, posicion, formato(valor), va="center",
                fontsize=12, color=TEXTO)

    ax.set_xlim(0, tope * 1.42)
    ax.set_ylim(-0.7, len(SERIES_GENERALES) - 0.3)
    ax.set_xticks([])
    ax.set_yticks(posiciones if panel == 0 else [])
    if panel == 0:
        ax.set_yticklabels([etiqueta for _, etiqueta, _ in SERIES_GENERALES], fontsize=12)
    ax.tick_params(axis="y", length=0, pad=10)
    sin_marco(ax)
    ax.set_title(etiqueta_de_carga(carga), fontsize=14, pad=14, color=TEXTO_TENUE)

figura.suptitle("Las dos consultas sumadas", fontsize=18, fontweight="bold",
                color=TEXTO, y=1.06)
figura.tight_layout(w_pad=3)
guardar(figura, "general.png")
plt.show()